In [0]:
# ==========================================================
# Project : Lyon Rental Market Analytics
# Layer   : Bronze
# Notebook: 01_ingest_loyers_bronze
#
# Description
# ----------------------------------------------------------
# This notebook ingests the raw Lyon rent regulation dataset 
# from Azure Data Lake Storage Gen2 into the Bronze layer.
#
# No transformation is performed except adding ingestion metadata.
#
# Input
# ----------------------------------------------------------
# CSV file stored in ADLS Gen2
#
# Output
# ----------------------------------------------------------
# Bronze Delta Table
#
# Author : Xin Yao
# ==========================================================


# ---------------------------------------
# 1. Import libraries
# ---------------------------------------

from pyspark.sql.functions import (
    current_timestamp,
    input_file_name,
    lit,
    col
)

# ---------------------------------------
# 2. Configuration
# ---------------------------------------

STORAGE_ACCOUNT = "stlyonloyersdev"
CONTAINER = "lyon-data"

SOURCE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "bronze/loyer/car_care.carencadrmtloyer_2023_2024"
)

TARGET_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    "bronze/delta/loyer"
)

# ---------------------------------------
# 3. Read raw CSV
# ---------------------------------------

print("Reading raw CSV...")

df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("delimiter", ",")
        .option("quote", '"')
        .option("escape", '"')
        .option("ignoreLeadingWhiteSpace", True)
        .option("ignoreTrailingWhiteSpace", True)
        .csv(SOURCE_PATH)
)

display(df.limit(5))

print(df.select("valeurs").first()["valeurs"])

# ---------------------------------------
# 4. Add ingestion metadata
# ---------------------------------------

print("Adding metadata...")

bronze_df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("data_source", lit("Encadrement des loyers"))
)

display(bronze_df.limit(5))

# ---------------------------------------
# 5. Save as Bronze Delta Table
# ---------------------------------------

print("Writing Bronze Delta Table...")

(
    bronze_df.write
             .format("delta")
             .mode("overwrite")
             .save(TARGET_PATH)
)

print("Bronze ingestion completed.")

# ---------------------------------------
# 6. Validation
# ---------------------------------------

validation_df = spark.read.format("delta").load(TARGET_PATH)

print(f"Number of records : {validation_df.count()}")

display(validation_df.limit(5))

In [0]:
print(bronze_df.select("valeurs").first()["valeurs"])

In [0]:
from pyspark.sql.functions import length

bronze_df.select(
    length("valeurs").alias("len"),
    "valeurs"
).show(1, truncate=False)